# Corroboration externe — collectes tierces, ligne à ligne

**Question** : deux collectes publiques *indépendantes* re-lisent les mêmes sources officielles
(les PTR du Sénat et de la Chambre). Retrouve-t-on **nos** lignes chez elles, transaction par transaction ?

- **senate-stock-watcher** — `data/external/senate_openset/ssw_all_daily_summaries.json`
- **house-stock-watcher**  — `data/external/house_openset/hsw_all_transactions.json`

**Clé d'appariement** : `déposant · ticker · sens · date` (le montant est hors clé — la loi ne
donne qu'une tranche). On indexe par `ticker` **et** `ticker_yahoo` (renommages), avec des clés-nom
robustes (suffixes Jr./III, noms composés). On rapporte le sous-ensemble `asset_type == Stock`,
seul directement comparable à notre table backtest (qui filtre les fonds mutuels et ETF exotiques).

**Principe** : ces sources ne servent qu'à *mesurer* — **jamais réinjectées** dans nos tables.
La concordance mesure la **robustesse de notre lecture** (deux lectures indépendantes qui convergent).

> Code réutilisable : `common/crosscheck.py` (`load_ssw_lines`, `load_hsw_lines`, `corroboration_lignes`).


In [1]:
import sys; sys.path.insert(0, '.')
import pandas as pd
from common import crosscheck as cc

BT = 'data/clean/transactions_backtest_2014_2026.csv'
df = pd.read_csv(BT, low_memory=False, dtype={c: str for c in
      ['ticker','ticker_yahoo','direction','transaction_date','member_name','chamber']})
print('table de recherche :', df.shape, '| Sénat:', int((df.chamber=="senate").sum()),
      '| Chambre:', int((df.chamber=="house").sum()))
df[['member_name','chamber','ticker','direction','transaction_date']].head(3)

table de recherche : (134464, 36) | Sénat: 11650 | Chambre: 122814


,member_name,chamber,ticker,direction,transaction_date
0,Bob Gibbs,house,INTC,buy,2014-12-30
1,Bob Gibbs,house,B,sell,2014-12-23
2,Bob Gibbs,house,AAPL,buy,2014-12-30


## Sénat — `senate-stock-watcher`


In [2]:
ssw   = cc.load_ssw_lines('data/external/senate_openset/ssw_all_daily_summaries.json')
r_ssw = cc.corroboration_lignes(df, ssw, 'senate')
a = r_ssw['actions']
print(f"actions cotées SSW : {a['n']}")
print(f"  retrouvées à la transaction près (déposant·ticker·sens·date) : {a['exact']} = {a['pct_exact']} %")
print(f"  toutes lignes cotées (fonds/ETF inclus) : {r_ssw['tout_cote']['pct_exact']} %"
      f"  — l'écart = fonds mutuels/ETF que notre pipeline filtre, pas des trous")
print(f"  échanges (SSW scinde en 2 lignes, représentés autrement) : {r_ssw['n_echanges']}")
print(f"  résidu d'actions non retrouvées : {len(r_ssw['residu'])} (tickers exotiques)")
r_ssw['residu'].head(10)

actions cotées SSW : 5524
  retrouvées à la transaction près (déposant·ticker·sens·date) : 5510 = 99.7 %
  toutes lignes cotées (fonds/ETF inclus) : 96.2 %  — l'écart = fonds mutuels/ETF que notre pipeline filtre, pas des trous
  échanges (SSW scinde en 2 lignes, représentés autrement) : 66
  résidu d'actions non retrouvées : 10 (tickers exotiques)


,ticker,sens,date
0,IBM.MX,buy,2018-04-24
1,NEE-PC,buy,2017-01-25
2,WFC,sell,2016-12-23
3,SYY.SG,sell,2016-05-11
4,XLS-WI,sell,2015-02-12
5,XLS-WI,sell,2015-03-03
6,XLS-WI,sell,2015-03-05
7,XLS-WI,sell,2015-03-09
8,XLS-WI,sell,2015-03-11
9,XLS-WI,sell,2015-03-16


## Chambre — `house-stock-watcher`


In [3]:
hsw   = cc.load_hsw_lines('data/external/house_openset/hsw_all_transactions.json')
r_hsw = cc.corroboration_lignes(df, hsw, 'house')
a = r_hsw['actions']
print(f"actions cotées HSW : {a['n']}")
print(f"  retrouvées à la transaction près : {a['exact']} = {a['pct_exact']} %")
print(f"  résidu d'actions non retrouvées : {len(r_hsw['residu'])} (surtout 2026, période très récente)")
r_hsw['residu'].head(10)

actions cotées HSW : 23437
  retrouvées à la transaction près : 23350 = 99.6 %
  résidu d'actions non retrouvées : 17 (surtout 2026, période très récente)


,ticker,sens,date
0,APH,buy,2026-07-08
1,MCK,buy,2026-07-08
2,HLT,buy,2026-06-15
3,SPHR,buy,2026-06-15
4,ESAB,buy,2026-06-22
5,ESAB,buy,2026-06-24
6,GIL,sell,2026-06-17
7,GIL,buy,2026-06-02
8,WPM,buy,2026-06-04
9,SPCX,buy,2026-06-15


## Synthèse


In [4]:
recap = pd.DataFrame([
 {'source':'senate-stock-watcher (Sénat)',  'lignes actions': r_ssw['actions']['n'],
  'retrouvées (date exacte)': f"{r_ssw['actions']['pct_exact']} %", 'résidu': len(r_ssw['residu'])},
 {'source':'house-stock-watcher (Chambre)', 'lignes actions': r_hsw['actions']['n'],
  'retrouvées (date exacte)': f"{r_hsw['actions']['pct_exact']} %", 'résidu': len(r_hsw['residu'])},
])
print('Deux collectes tierces indépendantes → >99,6 % de nos lignes d\'actions retrouvées, à la transaction près.')
recap

Deux collectes tierces indépendantes → >99,6 % de nos lignes d'actions retrouvées, à la transaction près.


,source,lignes actions,retrouvées (date exacte),résidu
0,senate-stock-watcher (Sénat),5524,99.7 %,10
1,house-stock-watcher (Chambre),23437,99.6 %,17
